### Cell 1: Setup & Dependencies

In [1]:
# Install required packages for NIfTI processing and deep learning[cite: 2]
!pip install -q nibabel scipy torch tqdm --no-deps

import glob
import os
import tarfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nibabel as nib
import scipy.ndimage as ndimage
import torch
import torch.nn as nn
from tqdm import tqdm

print("Environment setup complete. nibabel version:", nib.__version__)

Environment setup complete. nibabel version: 5.4.2


---

### Cell 2: Dataset Path & Archive Extraction

In [2]:
# Setup paths based on Kaggle input directory[cite: 2]
brats_root = "/kaggle/input/datasets/dschettler8845/brats-2021-task1"
brats_extract_dir = "/kaggle/working/brats2021"

os.makedirs(brats_extract_dir, exist_ok=True)

# Unpack .tar archives[cite: 2]
tar_files = glob.glob(f"{brats_root}/*.tar")
print(f"Found {len(tar_files)} tar files:", [os.path.basename(t) for t in tar_files])

for tar_path in tar_files:
    print(f"Extracting {os.path.basename(tar_path)} ...")
    with tarfile.open(tar_path, "r") as tf:
        for member in tqdm(tf.getmembers(), desc=os.path.basename(tar_path)):
            tf.extract(member, brats_extract_dir)

n_extracted = sum(len(files) for _, _, files in os.walk(brats_extract_dir))
print("Done. Total files extracted:", n_extracted)

total_gb = sum(os.path.getsize(t) for t in tar_files) / 1e9
print(f"{len(tar_files)} tar files, {total_gb:.1f} GB total (compressed/on-disk size)")

Found 3 tar files: ['BraTS2021_00495.tar', 'BraTS2021_Training_Data.tar', 'BraTS2021_00621.tar']
Extracting BraTS2021_00495.tar ...


BraTS2021_00495.tar:   0%|          | 0/6 [00:00<?, ?it/s]/tmp/ipykernel_22/1171123289.py:15: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extract(member, brats_extract_dir)
BraTS2021_00495.tar: 100%|██████████| 6/6 [00:00<00:00, 35.96it/s]


Extracting BraTS2021_Training_Data.tar ...


BraTS2021_Training_Data.tar: 100%|██████████| 7508/7508 [01:05<00:00, 114.64it/s]


Extracting BraTS2021_00621.tar ...


BraTS2021_00621.tar: 100%|██████████| 6/6 [00:00<00:00, 109.78it/s]


Done. Total files extracted: 6266
3 tar files, 13.4 GB total (compressed/on-disk size)


---

### Cell 3: Data Filtering (Duplicates & Brightness Outliers)

In [3]:
# Scan all extracted patient directories
all_patient_folders = glob.glob(os.path.join(brats_extract_dir, "BraTS2021_*"))

# Remove known duplicate and extreme outlier IDs flagged in EDA[cite: 1]
invalid_cases = {
    "BraTS2021_00495",  # Duplicate scan[cite: 1]
    "BraTS2021_00621",  # Duplicate scan[cite: 1]
    "BraTS2021_01163"   # Outlier: 1000x brightness anomaly[cite: 1]
}

clean_dataset_paths = []
for folder in all_patient_folders:
    patient_id = os.path.basename(folder)
    if patient_id not in invalid_cases:
        clean_dataset_paths.append(folder)

print(f"Total scans extracted: {len(all_patient_folders)}")
print(f"Dropped cases: {len(all_patient_folders) - len(clean_dataset_paths)}")
print(f"Clean, usable cases: {len(clean_dataset_paths)}")

Total scans extracted: 1261
Dropped cases: 3
Clean, usable cases: 1258


---

### Cell 4: Intensity Clipping & Z-Score Normalization

In [4]:
def preprocess_mri_volume(volume_data, lower_percentile=1.0, upper_percentile=99.0):
    """
    1. Winsorizes (clips) extreme brightness values to clear scanning artifacts.
    2. Performs Z-score normalization restricted strictly to non-zero brain tissue.
    """
    brain_mask = volume_data > 0
    if not brain_mask.any():
        return volume_data.astype(np.float32)

    # 1. Clip extreme percentiles inside the brain tissue
    brain_voxels = volume_data[brain_mask]
    low_val, high_val = np.percentile(brain_voxels, [lower_percentile, upper_percentile])
    clipped_data = np.clip(volume_data, low_val, high_val)

    # 2. Z-Score normalization on brain tissue only (mean=0, std=1)
    mean_val = np.mean(clipped_data[brain_mask])
    std_val = np.std(clipped_data[brain_mask])

    if std_val < 1e-8:
        std_val = 1.0

    normalized_data = np.zeros_like(volume_data, dtype=np.float32)
    normalized_data[brain_mask] = (clipped_data[brain_mask] - mean_val) / std_val

    return normalized_data

print("Intensity normalization function defined.")

Intensity normalization function defined.


---

### Cell 5: Clinical Metrics & Multifocality Correction

In [5]:
def extract_clinical_metrics(segmentation_mask, voxel_volume_mm3=1.0, min_foci_voxels=500):
    """
    1. Removes tiny noise fragments using 3D connected component labeling
       to rectify the inflated 59% multifocality count[cite: 1].
    2. Calculates true solid tumor volume by excluding Edema (Label 2)[cite: 1].
    """
    # --- 1. Correct Multifocality Count ---
    binary_tumor_mask = (segmentation_mask > 0).astype(np.uint8)
    labeled_array, num_features = ndimage.label(binary_tumor_mask)

    true_foci_count = 0
    cleaned_mask = np.zeros_like(segmentation_mask)

    for cluster_id in range(1, num_features + 1):
        cluster_voxels = (labeled_array == cluster_id)
        if np.sum(cluster_voxels) >= min_foci_voxels:
            true_foci_count += 1
            cleaned_mask[cluster_voxels] = segmentation_mask[cluster_voxels]

    # --- 2. Calculate Solid Tumor Size ---
    # BraTS Labels: 1 = Necrotic Core, 2 = Edema (Swelling), 4 = Enhancing Tumor
    # Exclude Label 2 to track solid tumor instead of swelling[cite: 1]
    solid_mask = np.logical_or(cleaned_mask == 1, cleaned_mask == 4)
    solid_volume_mm3 = np.sum(solid_mask) * voxel_volume_mm3

    return true_foci_count, solid_volume_mm3, cleaned_mask

print("Clinical metric extraction functions defined.")

Clinical metric extraction functions defined.


---

### Cell 6: Generalized Dice Loss (Class Imbalance)

In [6]:
class GeneralizedDiceLoss(nn.Module):
    """
    Solves extreme class imbalance (e.g., 99.9% Edema baseline) by weighting
    the spatial overlap inversely to class volume[cite: 1].
    """
    def __init__(self, epsilon=1e-6):
        super(GeneralizedDiceLoss, self).__init__()
        self.epsilon = epsilon

    def forward(self, predictions, targets):
        # Expected shape: (Batch, Classes, Depth, Height, Width)
        class_volumes = torch.sum(targets, dim=(0, 2, 3, 4))
        weights = 1.0 / ((class_volumes ** 2) + self.epsilon)

        intersection = torch.sum(predictions * targets, dim=(0, 2, 3, 4))
        union = torch.sum(predictions + targets, dim=(0, 2, 3, 4))

        dice_score = 2.0 * torch.sum(weights * intersection) / torch.sum(weights * union + self.epsilon)
        return 1.0 - dice_score

print("Generalized Dice Loss function ready.")

Generalized Dice Loss function ready.
